# 🌌 MURE-AGI: Autonomous Training Pipeline (1M Rules)
### Stage 1: vLLM Dataset Gen | Stage 2: QLoRA FineTune | Stage 3: GGUF Export
**Project Folder:** `/content/drive/MyDrive/mure_auto_train`

In [ ]:
# 1. 🌌 MURE-AGI Workspace & Drive Sync
from google.colab import drive
import os, sys, torch, gc

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=True)

ROOT_DIR = "/content/drive/MyDrive/mure_auto_train"
DATA_DIR = os.path.join(ROOT_DIR, "dataset")
MODEL_DIR = os.path.join(ROOT_DIR, "checkpoints")
GGUF_DIR = os.path.join(ROOT_DIR, "final_model_gguf")

for p in [DATA_DIR, MODEL_DIR, GGUF_DIR]:
    os.makedirs(p, exist_ok=True)

print(f"✅ Workspace: {ROOT_DIR}")
print(f"📌 Hardware: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (Slow)'}")

print("\n🔥 [IMPORTANT] Run this in Browser Console (F12 -> Console) to stay online:")
print('function ClickConnect(){ console.log("Auto-Keeping Session Alive"); document.querySelector("colab-connect-button").click(); } setInterval(ClickConnect, 60000);')


In [ ]:
# 2. Install Dependencies
%%capture
!pip install -q \"unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git\"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes
!pip install -q vllm jsonlines tqdm
print("🚀 Engines installed.")


In [ ]:
# 3. Dataset Generation (1M Rules) + Memory Flushing
import json, random, os, torch, gc
from tqdm.auto import tqdm

DATASET_PATH = os.path.join(DATA_DIR, "mure_1m_dataset.jsonl")
TARGET = 1_000_000
BATCH_SIZE = 2500
HAS_VRAM = torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 12e9

def get_count():
    if not os.path.exists(DATASET_PATH): return 0
    with open(DATASET_PATH, "rb") as f: return sum(1 for _ in f)

current_count = get_count()
if current_count < TARGET:
    model_id = "Qwen/Qwen2.5-1.5B-Instruct"
    if HAS_VRAM:
        from vllm import LLM, SamplingParams
        llm = LLM(model=model_id, max_model_len=512, enforce_eager=True, gpu_memory_utilization=0.5)
        params = SamplingParams(temperature=0.8, top_p=0.9, max_tokens=100)
        
        seeds = ["quantum computing", "neural pathway", "economic shift", "biological mutation", "social engineering"]
        pbar = tqdm(total=TARGET, initial=current_count, desc="⚡ Generating Logic")
        
        while current_count < TARGET:
            prompts = [f"Describe a complex causal chain for {random.choice(seeds)}. Cause -> Effect:" for _ in range(BATCH_SIZE)]
            outputs = llm.generate(prompts, params, use_tqdm=False)
            
            with open(DATASET_PATH, "a") as f:
                for out in outputs:
                    f.write(json.dumps({"instruction": "Analyze cause and effect", "input": "", "output": out.outputs[0].text.strip()}) + "\n")
                f.flush(); os.fsync(f.fileno())
            current_count += BATCH_SIZE
            pbar.update(BATCH_SIZE)
            
        # CRITICAL: Clean memory for training
        del llm
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
        print("🧹 Memory cleared for Training Stage.")
    else:
        print("⚠️ GPU memory low or missing. Generation skipped for safety (manual upload recommended if local).")
else:
print("✅ 1M Rule Dataset Ready.")
import gc
if 'llm' in locals():
    del llm
    gc.collect(); torch.cuda.empty_cache()
    print("🧹 vLLM engine cleared for Training.")


In [ ]:
# 4. Unsloth QLoRA (4-bit) Finetuning
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-3B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(
    model, r = 16, target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16, lora_dropout = 0, bias = "none",
)

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
def formatting_func(examples):
    texts = []
    for inst, out in zip(examples.get("instruction", []), examples.get("output", [])):
        texts.append(f"<|im_start|>user\n{inst}<|im_end|>\n<|im_start|>assistant\n{out}<|im_end|>")
    return {"text": texts}
dataset = dataset.map(formatting_func, batched=True)

trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 1024,
    args = TrainingArguments(
        per_device_train_batch_size = 4, gradient_accumulation_steps = 4,
        warmup_steps = 100, max_steps = 1000, learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(), bf16 = torch.cuda.is_bf16_supported(),
        output_dir = MODEL_DIR, save_steps = 500,
    ),
)
print("💪 Starting Training...")
trainer.train()


In [ ]:
# 5. GGUF Export (Universal MURE Brain Format)
print("🛠️ Starting GGUF Conversion (Quantization: Q4_K_M)")

try:
    model.save_pretrained_gguf(
        GGUF_DIR,
        tokenizer,
        quantization_method = "q4_k_m", # Best balance of speed and logic retention
    )
    print(f"⭐ SUCCESS: Exported to {GGUF_DIR}")
    print("📂 You can now download the .gguf file from your Google Drive for deployment.")
except Exception as e:
    print(f"❌ Export failure: {e}")
    print("Falling back to standard LoRA save...")
    model.save_pretrained(os.path.join(ROOT_DIR, 'MURE_3B_LoRA_Backup'))
